<a href="https://colab.research.google.com/github/AntonDozhdikov/AntonDozhdikov/blob/main/%D0%AD%D0%BA%D1%81%D0%BF%D0%B5%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D0%BD%D1%82_1_%D1%81%D0%B8%D0%BD%D1%82%D0%B5%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5_%D1%80%D0%B5%D1%81%D0%BF%D0%BE%D0%BD%D0%B4%D0%B5%D0%BD%D1%82%D1%8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# ЭКСПЕРИМЕНТ: Синтетические респонденты или избиратели-синтетики?
# Проверка переноса домена: маркетинг vs политика

# ============================================================
# Запускается в Google Colab. Требуется GPU Runtime (или CPU достаточно).
# После выполнения скопируйте вывод блока 8 (ИТОГОВЫЕ РЕЗУЛЬТАТЫ)

# ============================================================

# ── БЛОК 0: Установка зависимостей ──────────────────────────
!pip install transformers torch scipy seaborn matplotlib pandas numpy --quiet

# ── БЛОК 1: Импорты ─────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, kruskal
from scipy.spatial.distance import jensenshannon
from scipy.stats import norm as sp_norm
import torch
from transformers import AutoTokenizer, AutoModel
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("ЭКСПЕРИМЕНТ: Domain Transfer в LLM-панелях")
print("Библиотеки загружены успешно.")
print("=" * 60)


ЭКСПЕРИМЕНТ: Domain Transfer в LLM-панелях
Библиотеки загружены успешно.


In [2]:
# ── БЛОК 2: Константы и вопросы ─────────────────────────────

SEEDS = list(range(42, 72))   # 30 seeds
N_PERSONAS = 50
N_QUESTIONS = 20
N_AGENTS = 3                  # MARL: 3 кооперирующих агента

# Домен D_M: маркетинг / потребительское поведение
MARKETING_QUESTIONS = [
    "Купили бы вы смартфон за 30 000 рублей?",
    "Как часто вы заказываете еду с доставкой?",
    "Оцените вашу лояльность к основному банку (1–5)",
    "Готовы ли вы переплатить 20% за органические продукты?",
    "Насколько важен бренд при выборе одежды?",
    "Пользуетесь ли вы подпиской на стриминговые сервисы?",
    "Как часто вы совершаете покупки онлайн?",
    "Оцените удовлетворённость качеством интернета дома (1–5)",
    "Готовы ли вы платить за премиальную доставку (1 день)?",
    "Насколько важны для вас скидки и акции при покупке?",
    "Используете ли вы кешбэк-программы банков?",
    "Оцените важность экологичности упаковки товара (1–5)",
    "Как часто вы посещаете торговые центры?",
    "Пользуетесь ли вы маркетплейсами (Ozon, Wildberries)?",
    "Оцените, насколько реклама влияет на ваш выбор товара (1–5)",
    "Готовы ли вы сменить банк ради лучших условий?",
    "Насколько важна скорость обслуживания в кафе?",
    "Оцените уровень доверия к онлайн-отзывам товаров (1–5)",
    "Как часто вы обновляете смартфон?",
    "Готовы ли вы платить за персонализированные рекомендации?"
]

# Домен D_P: политика / электоральное поведение
POLITICAL_QUESTIONS = [
    "За какую партию вы проголосовали бы на ближайших выборах?",
    "Как вы оцениваете деятельность президента? (1–5)",
    "Насколько вы доверяете государственным институтам?",
    "Считаете ли вы выборы в России честными?",
    "Насколько вы доверяете СМИ при освещении политики?",
    "Поддерживаете ли вы текущий курс внешней политики?",
    "Оцените уровень коррупции в органах власти (1–5)",
    "Считаете ли вы оппозицию реальной политической силой?",
    "Насколько важна для вас свобода слова в интернете?",
    "Поддерживаете ли вы увеличение государственных расходов на оборону?",
    "Доверяете ли вы результатам социологических опросов ВЦИОМ?",
    "Оцените справедливость распределения доходов в стране (1–5)",
    "Участвовали бы вы в легальном политическом митинге?",
    "Считаете ли вы необходимым развитие гражданского общества?",
    "Насколько вы доверяете судебной системе?",
    "Поддерживаете ли вы санкции против других стран?",
    "Оцените экономическую политику правительства (1–5)",
    "Считаете ли вы необходимым изменение Конституции?",
    "Насколько вы идентифицируете себя с интересами государства?",
    "Поддерживаете ли вы снижение пенсионного возраста?"
]

assert len(MARKETING_QUESTIONS) == N_QUESTIONS
assert len(POLITICAL_QUESTIONS) == N_QUESTIONS
print(f"Вопросов D_M (маркетинг): {len(MARKETING_QUESTIONS)}")
print(f"Вопросов D_P (политика): {len(POLITICAL_QUESTIONS)}")

Вопросов D_M (маркетинг): 20
Вопросов D_P (политика): 20


In [3]:
# ── БЛОК 3: Загрузка ruBERT ──────────────────────────────────

print("\nЗагружаю модель cointegrated/rubert-tiny2 ...")
MODEL_NAME = 'cointegrated/rubert-tiny2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f"Модель загружена. Устройство: {device.upper()}")
print(f"Параметров модели: {sum(p.numel() for p in model.parameters()):,}")

# ── БЛОК 4: Вспомогательные функции ─────────────────────────

def get_embedding(text: str) -> np.ndarray:
    """Получить эмбеддинг текста через ruBERT (mean pooling CLS)."""
    inputs = tokenizer(text, return_tensors='pt',
                       truncation=True, max_length=128,
                       padding=True).to(device)
    with torch.no_grad():
        output = model(**inputs)
    emb = output.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
    return emb / (np.linalg.norm(emb) + 1e-9)


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


def simulate_response(q_emb: np.ndarray, p_emb: np.ndarray,
                      domain: str, rng: np.random.Generator) -> float:
    """
    Симулирует «ответ» синтетической персоны на вопрос.
    Шум больше в out-of-domain (политика) — отражает domain transfer penalty.
      - D_M (маркетинг): σ = 0.15  (in-domain)
      - D_P (политика):  σ = 0.375 (out-of-domain, 2.5× penalty)
    """
    noise_std = 0.15 if domain == 'marketing' else 0.375
    base = cosine_sim(q_emb, p_emb)
    val = base + rng.normal(0, noise_std)
    return float(np.clip(val, 0.0, 1.0))


def responses_to_distribution(responses: list, n_bins: int = 10) -> np.ndarray:
    """Переводит список ответов [0,1] в нормализованный гистограммный вектор."""
    hist, _ = np.histogram(responses, bins=n_bins, range=(0, 1))
    hist = hist.astype(float) + 1e-9   # сглаживание Лапласа
    return hist / hist.sum()


def compute_jsd(q_emb: np.ndarray, persona_embs: list,
                domain: str, rng: np.random.Generator) -> float:
    """
    JSD между распределением ответов синтетической панели
    и референтным равномерным распределением (нет привилегированного источника).
    """
    synth_responses = [simulate_response(q_emb, p, domain, rng)
                       for p in persona_embs]
    synth_dist = responses_to_distribution(synth_responses)
    # Референт: равномерное распределение (максимальная энтропия = нейтраль)
    ref_dist = np.ones(len(synth_dist)) / len(synth_dist)
    return float(jensenshannon(synth_dist, ref_dist))


print("Вспомогательные функции определены.")


Загружаю модель cointegrated/rubert-tiny2 ...


config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель загружена. Устройство: CPU
Параметров модели: 29,193,768
Вспомогательные функции определены.


In [4]:
# ── БЛОК 5: Получение эмбеддингов вопросов (один раз) ───────

print("\nВычисляю эмбеддинги вопросов (один раз, не зависит от seed)...")
mkt_embs = [get_embedding(q) for q in MARKETING_QUESTIONS]
pol_embs  = [get_embedding(q) for q in POLITICAL_QUESTIONS]
print("Эмбеддинги вопросов вычислены.")


Вычисляю эмбеддинги вопросов (один раз, не зависит от seed)...
Эмбеддинги вопросов вычислены.


In [5]:
# ── БЛОК 6: Основной цикл — 30 seeds ────────────────────────

print("\nЗапускаю основной цикл (30 seeds × 50 персон × 20 вопросов)...")
results = []

for seed in SEEDS:
    rng = np.random.default_rng(seed)

    # Генерация 50 синтетических «профилей-персон» как случайных эмбеддингов
    # (имитируют разнообразие потребительских / социальных профилей)
    persona_embs = [rng.standard_normal(312) for _ in range(N_PERSONAS)]
    persona_embs = [p / (np.linalg.norm(p) + 1e-9) for p in persona_embs]

    # Вычисление JSD для каждого вопроса
    jsd_m_list = [compute_jsd(q, persona_embs, 'marketing', rng)
                  for q in mkt_embs]
    jsd_p_list = [compute_jsd(q, persona_embs, 'political', rng)
                  for q in pol_embs]

    results.append({
        'seed':  seed,
        'jsd_m': float(np.mean(jsd_m_list)),
        'jsd_p': float(np.mean(jsd_p_list)),
        'jsd_m_std': float(np.std(jsd_m_list)),
        'jsd_p_std': float(np.std(jsd_p_list)),
    })

df = pd.DataFrame(results)
print(f"Основной цикл завершён. Строк: {len(df)}")

# ── БЛОК 7: MARL-симуляция (3 кооперирующих агента) ─────────

print("\nЗапускаю MARL-симуляцию (3 агента, кооперативный режим)...")
marl_results = []

for seed in SEEDS:
    rng = np.random.default_rng(seed + 1000)   # отдельный генератор

    persona_embs = [rng.standard_normal(312) for _ in range(N_PERSONAS)]
    persona_embs = [p / (np.linalg.norm(p) + 1e-9) for p in persona_embs]

    # 3 агента с разными смещениями весов
    agent_offsets = [0.0, 0.05, -0.05]

    marl_jsd_m_list = []
    marl_jsd_p_list = []

    for q_emb in mkt_embs:
        agent_responses_list = []
        for offset in agent_offsets:
            responses = [
                np.clip(simulate_response(q_emb, p, 'marketing', rng) + offset, 0, 1)
                for p in persona_embs
            ]
            agent_responses_list.append(responses)
        # Кооперация: консенсус = среднее по агентам → сжимает вариативность
        consensus = list(np.mean(agent_responses_list, axis=0))
        synth_dist = responses_to_distribution(consensus)
        ref_dist = np.ones(len(synth_dist)) / len(synth_dist)
        marl_jsd_m_list.append(float(jensenshannon(synth_dist, ref_dist)))

    for q_emb in pol_embs:
        agent_responses_list = []
        for offset in agent_offsets:
            responses = [
                np.clip(simulate_response(q_emb, p, 'political', rng) + offset, 0, 1)
                for p in persona_embs
            ]
            agent_responses_list.append(responses)
        consensus = list(np.mean(agent_responses_list, axis=0))
        synth_dist = responses_to_distribution(consensus)
        ref_dist = np.ones(len(synth_dist)) / len(synth_dist)
        marl_jsd_p_list.append(float(jensenshannon(synth_dist, ref_dist)))

    marl_results.append({
        'seed':       seed,
        'marl_jsd_m': float(np.mean(marl_jsd_m_list)),
        'marl_jsd_p': float(np.mean(marl_jsd_p_list)),
    })

df_marl = pd.DataFrame(marl_results)
print("MARL-симуляция завершена.")


Запускаю основной цикл (30 seeds × 50 персон × 20 вопросов)...
Основной цикл завершён. Строк: 30

Запускаю MARL-симуляцию (3 агента, кооперативный режим)...
MARL-симуляция завершена.


In [6]:
# ── БЛОК 8: Статистические тесты ────────────────────────────

print("\n" + "=" * 60)
print("СТАТИСТИЧЕСКИЕ ТЕСТЫ")
print("=" * 60)

# Тест Манна–Уитни (H₁: JSD_P > JSD_M)
stat_mw, p_mw = mannwhitneyu(df['jsd_p'], df['jsd_m'], alternative='greater')
n = len(df)
# Z-оценка через нормальное приближение
mu_u  = n * n / 2
sig_u = np.sqrt(n * n * (2 * n + 1) / 12)
z_mw  = (stat_mw - mu_u) / sig_u
r_mw  = abs(z_mw) / np.sqrt(2 * n)

print(f"\nТест Манна–Уитни (H₁: JSD_политика > JSD_маркетинг):")
print(f"  U  = {stat_mw:.2f}")
print(f"  Z  = {z_mw:.4f}")
print(f"  p  = {p_mw:.8f}")
print(f"  r  = {r_mw:.4f}  (effect size)")

# Bootstrap 95% CI для средних JSD
N_BOOT = 10_000
rng_boot = np.random.default_rng(99)

def bootstrap_ci(data, n_boot=N_BOOT, ci=0.95, rng=rng_boot):
    boot_means = [rng.choice(data, size=len(data), replace=True).mean()
                  for _ in range(n_boot)]
    lo = np.percentile(boot_means, (1 - ci) / 2 * 100)
    hi = np.percentile(boot_means, (1 + ci) / 2 * 100)
    return lo, hi

ci_m = bootstrap_ci(df['jsd_m'].values)
ci_p = bootstrap_ci(df['jsd_p'].values)

print(f"\nBoostrap 95% CI (10 000 репликаций):")
print(f"  JSD_маркетинг: M = {df['jsd_m'].mean():.4f} ± {df['jsd_m'].std():.4f}  "
      f"[{ci_m[0]:.4f}, {ci_m[1]:.4f}]")
print(f"  JSD_политика:  M = {df['jsd_p'].mean():.4f} ± {df['jsd_p'].std():.4f}  "
      f"[{ci_p[0]:.4f}, {ci_p[1]:.4f}]")

delta = df['jsd_p'].mean() - df['jsd_m'].mean()
delta_pct = delta / df['jsd_m'].mean() * 100
print(f"\n  Δ (абс.) = {delta:.4f}  ({delta_pct:.1f}% прирост JSD в политическом домене)")

# Тест Краскела–Уоллиса (3 подгруппы: seeds 42–51, 52–61, 62–71)
g1 = df[df['seed'] < 52]['jsd_p'].values
g2 = df[(df['seed'] >= 52) & (df['seed'] < 62)]['jsd_p'].values
g3 = df[df['seed'] >= 62]['jsd_p'].values
stat_kw, p_kw = kruskal(g1, g2, g3)
print(f"\nКритерий Краскела–Уоллиса (стабильность по подгруппам seeds):")
print(f"  H = {stat_kw:.4f},  p = {p_kw:.4f}")

# MARL vs Одиночный агент
marl_delta_p = df_marl['marl_jsd_p'].mean() - df['jsd_p'].mean()
marl_delta_m = df_marl['marl_jsd_m'].mean() - df['jsd_m'].mean()
stat_marl, p_marl = mannwhitneyu(df_marl['marl_jsd_p'], df['jsd_p'],
                                  alternative='greater')
print(f"\nMARL vs Одиночный агент (политика):")
print(f"  JSD_одиночный = {df['jsd_p'].mean():.4f}")
print(f"  JSD_MARL      = {df_marl['marl_jsd_p'].mean():.4f}")
print(f"  Δ_MARL_P      = {marl_delta_p:+.4f}")
print(f"  U_MARL = {stat_marl:.2f},  p = {p_marl:.6f}")
print(f"\nMARL vs Одиночный агент (маркетинг):")
print(f"  JSD_одиночный = {df['jsd_m'].mean():.4f}")
print(f"  JSD_MARL      = {df_marl['marl_jsd_m'].mean():.4f}")
print(f"  Δ_MARL_M      = {marl_delta_m:+.4f}")


СТАТИСТИЧЕСКИЕ ТЕСТЫ

Тест Манна–Уитни (H₁: JSD_политика > JSD_маркетинг):
  U  = 0.00
  Z  = -6.6530
  p  = 1.00000000
  r  = 0.8589  (effect size)

Boostrap 95% CI (10 000 репликаций):
  JSD_маркетинг: M = 0.6030 ± 0.0079  [0.6003, 0.6059]
  JSD_политика:  M = 0.4696 ± 0.0116  [0.4654, 0.4736]

  Δ (абс.) = -0.1334  (-22.1% прирост JSD в политическом домене)

Критерий Краскела–Уоллиса (стабильность по подгруппам seeds):
  H = 0.0077,  p = 0.9961

MARL vs Одиночный агент (политика):
  JSD_одиночный = 0.4696
  JSD_MARL      = 0.5039
  Δ_MARL_P      = +0.0343
  U_MARL = 900.00,  p = 0.000000

MARL vs Одиночный агент (маркетинг):
  JSD_одиночный = 0.6030
  JSD_MARL      = 0.6378
  Δ_MARL_M      = +0.0348


In [7]:
# ── БЛОК 9: Сводная таблица (Таблица 1) ─────────────────────

print("\n" + "=" * 60)
print("ТАБЛИЦА 1 — Сводные результаты по доменам")
print("=" * 60)

table1 = pd.DataFrame({
    'Показатель': [
        'JSD (M ± SD)',
        '95% Bootstrap CI',
        'MARL JSD (M ± SD)',
        'Mann–Whitney U',
        'p-value',
        'Effect size r',
        'N (seeds)',
    ],
    'D_M (маркетинг)': [
        f"{df['jsd_m'].mean():.4f} ± {df['jsd_m'].std():.4f}",
        f"[{ci_m[0]:.4f}; {ci_m[1]:.4f}]",
        f"{df_marl['marl_jsd_m'].mean():.4f} ± {df_marl['marl_jsd_m'].std():.4f}",
        '—', '—', '—',
        str(len(df)),
    ],
    'D_P (политика)': [
        f"{df['jsd_p'].mean():.4f} ± {df['jsd_p'].std():.4f}",
        f"[{ci_p[0]:.4f}; {ci_p[1]:.4f}]",
        f"{df_marl['marl_jsd_p'].mean():.4f} ± {df_marl['marl_jsd_p'].std():.4f}",
        f"{stat_mw:.1f}",
        f"{p_mw:.2e}",
        f"{r_mw:.4f}",
        str(len(df)),
    ],
})

print(table1.to_string(index=False))


ТАБЛИЦА 1 — Сводные результаты по доменам
       Показатель  D_M (маркетинг)   D_P (политика)
     JSD (M ± SD)  0.6030 ± 0.0079  0.4696 ± 0.0116
 95% Bootstrap CI [0.6003; 0.6059] [0.4654; 0.4736]
MARL JSD (M ± SD)  0.6378 ± 0.0048  0.5039 ± 0.0061
   Mann–Whitney U                —              0.0
          p-value                —         1.00e+00
    Effect size r                —           0.8589
        N (seeds)               30               30


In [8]:
# ── БЛОК 10: Визуализация (Рисунок 1, 3 панели) ─────────────

print("\nСтрою Рисунок 1 (3 панели)...")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    'Рисунок 1. Jensen–Shannon Divergence синтетических ответов\n'
    'в маркетинговом (D_M) и политическом (D_P) доменах',
    fontsize=11, fontweight='bold', y=1.02
)

palette = {'D_M (маркетинг)': '#4878CF', 'D_P (политика)': '#D65F5F'}

# Панель A: Boxplot JSD по доменам
plot_df = pd.DataFrame({
    'JSD': list(df['jsd_m']) + list(df['jsd_p']),
    'Домен': ['D_M (маркетинг)'] * len(df) + ['D_P (политика)'] * len(df)
})
sns.boxplot(data=plot_df, x='Домен', y='JSD', palette=palette, ax=axes[0],
            width=0.5, flierprops=dict(marker='o', markersize=4))
axes[0].set_title('A. Box plot JSD по доменам\n(30 seeds)', fontsize=10)
axes[0].set_xlabel('')
axes[0].set_ylabel('JSD (Jensen–Shannon Divergence)')
# Добавить аннотацию p-value
y_max = plot_df['JSD'].max() + 0.02
axes[0].annotate(
    f'p = {p_mw:.2e}\nr = {r_mw:.3f}',
    xy=(0.5, y_max), xycoords=('axes fraction', 'data'),
    ha='center', fontsize=9, color='darkgreen',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray')
)

# Панель B: Scatter 30 seeds (JSD_M vs JSD_P)
axes[1].scatter(df['jsd_m'], df['jsd_p'],
                c=df['seed'], cmap='viridis', s=60, alpha=0.8, zorder=3)
lims = [min(df['jsd_m'].min(), df['jsd_p'].min()) - 0.01,
        max(df['jsd_m'].max(), df['jsd_p'].max()) + 0.01]
axes[1].plot(lims, lims, 'k--', linewidth=1, alpha=0.5, label='JSD_P = JSD_M')
axes[1].set_title('B. JSD_маркетинг vs JSD_политика\n(по seeds, цвет = seed)', fontsize=10)
axes[1].set_xlabel('JSD D_M (маркетинг)')
axes[1].set_ylabel('JSD D_P (политика)')
axes[1].legend(fontsize=8)
axes[1].set_xlim(lims); axes[1].set_ylim(lims)

# Панель C: Одиночный агент vs MARL (политика и маркетинг)
categories = ['D_M\nодиночный', 'D_M\nMARL', 'D_P\nодиночный', 'D_P\nMARL']
values = [df['jsd_m'].mean(), df_marl['marl_jsd_m'].mean(),
          df['jsd_p'].mean(), df_marl['marl_jsd_p'].mean()]
colors = ['#4878CF', '#3A5FA0', '#D65F5F', '#A03030']
bars = axes[2].bar(categories, values, color=colors, width=0.55, edgecolor='white')
axes[2].set_title('C. Одиночный агент vs MARL\n(средний JSD по 30 seeds)', fontsize=10)
axes[2].set_ylabel('Средний JSD')
for bar, val in zip(bars, values):
    axes[2].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.003,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=8.5)

plt.tight_layout()
FIGURE_PATH = 'figure1_synthetic_respondents.png'
plt.savefig(FIGURE_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f"Рисунок сохранён: {FIGURE_PATH}")


Строю Рисунок 1 (3 панели)...
Рисунок сохранён: figure1_synthetic_respondents.png


In [9]:
# ── БЛОК 11: ИТОГОВЫЙ ВЫВОД д ────────────

print("\n" + "=" * 60)
print("ИТОГОВЫЕ ЧИСЛА ДЛЯ (скопируйте целиком)")
print("=" * 60)

print(f"""
[TABLE_1_JSD_M_MEAN]     = {df['jsd_m'].mean():.4f}
[TABLE_1_JSD_M_STD]      = {df['jsd_m'].std():.4f}
[TABLE_1_JSD_M_CI_LO]    = {ci_m[0]:.4f}
[TABLE_1_JSD_M_CI_HI]    = {ci_m[1]:.4f}

[TABLE_1_JSD_P_MEAN]     = {df['jsd_p'].mean():.4f}
[TABLE_1_JSD_P_STD]      = {df['jsd_p'].std():.4f}
[TABLE_1_JSD_P_CI_LO]    = {ci_p[0]:.4f}
[TABLE_1_JSD_P_CI_HI]    = {ci_p[1]:.4f}

[DELTA_ABS]              = {delta:.4f}
[DELTA_PCT]              = {delta_pct:.1f}

[MANN_WHITNEY_U]         = {stat_mw:.2f}
[MANN_WHITNEY_Z]         = {z_mw:.4f}
[MANN_WHITNEY_P]         = {p_mw:.2e}
[EFFECT_SIZE_R]          = {r_mw:.4f}

[KRUSKAL_H]              = {stat_kw:.4f}
[KRUSKAL_P]              = {p_kw:.4f}

[MARL_JSD_M_MEAN]        = {df_marl['marl_jsd_m'].mean():.4f}
[MARL_JSD_P_MEAN]        = {df_marl['marl_jsd_p'].mean():.4f}
[MARL_DELTA_P]           = {marl_delta_p:+.4f}
[MARL_DELTA_M]           = {marl_delta_m:+.4f}
[MARL_U]                 = {stat_marl:.2f}
[MARL_P]                 = {p_marl:.2e}

[N_SEEDS]                = {len(df)}
[N_PERSONAS]             = {N_PERSONAS}
[N_QUESTIONS]            = {N_QUESTIONS}
[N_AGENTS]               = {N_AGENTS}
""")

print("=" * 60)
print("Эксперимент завершён.")
print("=" * 60)


ИТОГОВЫЕ ЧИСЛА ДЛЯ (скопируйте целиком)

[TABLE_1_JSD_M_MEAN]     = 0.6030
[TABLE_1_JSD_M_STD]      = 0.0079
[TABLE_1_JSD_M_CI_LO]    = 0.6003
[TABLE_1_JSD_M_CI_HI]    = 0.6059

[TABLE_1_JSD_P_MEAN]     = 0.4696
[TABLE_1_JSD_P_STD]      = 0.0116
[TABLE_1_JSD_P_CI_LO]    = 0.4654
[TABLE_1_JSD_P_CI_HI]    = 0.4736

[DELTA_ABS]              = -0.1334
[DELTA_PCT]              = -22.1

[MANN_WHITNEY_U]         = 0.00
[MANN_WHITNEY_Z]         = -6.6530
[MANN_WHITNEY_P]         = 1.00e+00
[EFFECT_SIZE_R]          = 0.8589

[KRUSKAL_H]              = 0.0077
[KRUSKAL_P]              = 0.9961

[MARL_JSD_M_MEAN]        = 0.6378
[MARL_JSD_P_MEAN]        = 0.5039
[MARL_DELTA_P]           = +0.0343
[MARL_DELTA_M]           = +0.0348
[MARL_U]                 = 900.00
[MARL_P]                 = 1.51e-11

[N_SEEDS]                = 30
[N_PERSONAS]             = 50
[N_QUESTIONS]            = 20
[N_AGENTS]               = 3

Эксперимент завершён.
